# Paso 4 mejorado: Cruce real con datos climáticos Open-Meteo

## Problema potencial encontrado

¿Cómo influyen las condiciones meteorológicas en la demanda diaria del sistema Hubway?

El objetivo es complementar el análisis original incorporando una fuente externa de datos climáticos y evaluar relaciones entre temperatura, precipitación y cantidad de viajes.


In [ ]:
import requests

# Coordenadas aproximadas del centro de Boston
latitude = 42.355
longitude = -71.065

params = {
    'latitude': latitude,
    'longitude': longitude,
    'start_date': '2011-07-28',
    'end_date': '2013-11-30',
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum',
    'timezone': 'America/New_York'
}

response = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params)
weather = pd.DataFrame(response.json()['daily'])
weather['time'] = pd.to_datetime(weather['time']).dt.date
weather.head()


## Preparación de viajes diarios

Se agregan los viajes por fecha para poder unirlos con la información climática.


In [ ]:
trips['fecha'] = trips['start_dt'].dt.date

daily_trips = (
    trips.groupby('fecha')
    .size()
    .reset_index(name='n_viajes')
)

clima_viajes = daily_trips.merge(
    weather,
    left_on='fecha',
    right_on='time',
    how='inner'
)

clima_viajes.head()


## Relación entre clima y demanda

Se evalúan posibles relaciones:

- Temperatura máxima vs cantidad de viajes.
- Precipitación vs cantidad de viajes.


In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))

sns.scatterplot(
    data=clima_viajes,
    x='temperature_2m_max',
    y='n_viajes',
    alpha=0.4,
    ax=axes[0]
)
axes[0].set_title('Temperatura máxima vs viajes diarios')
axes[0].set_xlabel('Temperatura °C')

sns.scatterplot(
    data=clima_viajes,
    x='precipitation_sum',
    y='n_viajes',
    alpha=0.4,
    ax=axes[1]
)
axes[1].set_title('Precipitación vs viajes diarios')
axes[1].set_xlabel('Precipitación (mm)')

plt.tight_layout()
plt.show()


In [ ]:
print('Correlación temperatura-viajes:',
      clima_viajes['n_viajes'].corr(clima_viajes['temperature_2m_max']))

print('Correlación lluvia-viajes:',
      clima_viajes['n_viajes'].corr(clima_viajes['precipitation_sum']))


## Interpretación

El cruce con datos climáticos permite detectar que la demanda del sistema no depende únicamente del calendario, sino también de factores ambientales.

Una relación positiva con temperatura indicaría mayor uso durante días favorables, mientras que una relación negativa con precipitación sugeriría menor utilización durante lluvia.

Este análisis mejora el Paso 4 porque incorpora una dimensión externa y permite construir modelos predictivos de demanda.